In [ ]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report


pd.set_option('display.max_colwidth', None)
df = pd.read_csv("../data/clickbait_title_classification.csv")


def clean_text(text: str):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

df['clean_title'] = df['title'].apply(clean_text)


vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['clean_title'])
Y = df['clickbait']
print(X)




X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print("data processed")
print(f"training size: {X_train.shape}")
print(f"test size: {X_test.shape}")

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 243814 stored elements and shape (32000, 5000)>
  Coords	Values
  (0, 404)	0.49304192409185266
  (0, 1400)	0.5313195446718534
  (0, 2990)	0.4068950135594124
  (0, 3113)	0.2046446349891681
  (0, 3123)	0.48765677574280303
  (0, 3084)	0.1713391167254793
  (1, 2273)	0.8838346063940309
  (1, 3535)	0.4677995174648094
  (2, 4415)	0.5014218525626079
  (2, 3946)	0.4320389235503451
  (2, 1014)	0.48532411631462663
  (2, 4520)	0.1662687025934922
  (2, 302)	0.3221315864316587
  (2, 1540)	0.44154836201085373
  (3, 4520)	0.16927733399442071
  (3, 2349)	0.2393730592293937
  (3, 4982)	0.23228430796196856
  (3, 3012)	0.2735678349115182
  (3, 2620)	0.3411063434822727
  (3, 1938)	0.5663723474620141
  (3, 4445)	0.5900162617707052
  (4, 2349)	0.1994119735541915
  (4, 2819)	0.4203248382324195
  (4, 4450)	0.1458183189539087
  (4, 3119)	0.30777306120901815
  :	:
  (31995, 3153)	0.5040273724946985
  (31995, 1000)	0.5381099065796318
  (31996, 4520)	0.

In [20]:
df.head()

,title,clickbait,clean_title
0,""".asia"" domain applications near 300,000 on opening of registration",0,asia domain applications near 300000 on opening of registration
1,"""1 Indian + 1 Indian = Unrelatable"": Television's Race Equations",1,1 indian 1 indian unrelatable televisions race equations
2,"""7th Heaven"" television series comes to an end",0,7th heaven television series comes to an end
3,"""Arm Glow"" Is Your New Life Goal, Thanks To Lupita Nyong'o",1,arm glow is your new life goal thanks to lupita nyongo
4,"""Beans Memes"" Is The Only Twitter Account That Actually Matters",1,beans memes is the only twitter account that actually matters


In [15]:
model = XGBClassifier()
model.fit(X_train,Y_train)

print(f"model trained")

model trained


In [16]:
Y_pred = model.predict(X_test)
print(Y_pred)

[0 0 0 ... 0 1 1]


In [17]:
accuracy = accuracy_score(Y_test,Y_pred)
print(f"Accuracy: {accuracy:.2%}")

print(classification_report(Y_test,Y_pred))

Accuracy: 94.56%
              precision    recall  f1-score   support

           0       0.92      0.97      0.95      3193
           1       0.97      0.92      0.94      3207

    accuracy                           0.95      6400
   macro avg       0.95      0.95      0.95      6400
weighted avg       0.95      0.95      0.95      6400



In [18]:
import pickle

with open('../models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer,f)
    
    model.save_model('../models/xgboost_clickbait.json')
    
print('model saved')

model saved


#Edge Cases where the model struggles(these could go either way)

In [ ]:

edge_cases = [
    "Why Scientists Are Worried About Arctic Ice",
    "How This Teacher Changed One Student's Life",
    "What Doctors Don't Tell You About Sleep"
]

for title in edge_cases:
    cleaned = clean_text(title)
    vectorized = vectorizer.transform([cleaned])
    pred = model.predict(vectorized)[0]
    prob = model.predict_proba(vectorized)[0]
    label = "CLICKBAIT" if pred == 1 else "NOT CLICKBAIT"
    confidence = prob[1] if pred == 1 else prob[0]
    print(f"{label} ({confidence:.0%}): {title}")
    
    print(vectorized.get_shape())
    
    print(vectorized)

    sim_vector = X@vectorized.T
    print(sim_vector.get_shape())
    sim_idx = np.argmax(sim_vector)
    print(sim_idx)
    
    

CLICKBAIT (88%): Why Scientists Are Worried About Arctic Ice
(1, 5000)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6 stored elements and shape (1, 5000)>
  Coords	Values
  (0, 130)	0.26986673738772293
  (0, 369)	0.5501620339101009
  (0, 370)	0.23302907229366404
  (0, 2221)	0.45463973200594243
  (0, 3874)	0.472607654026108
  (0, 4878)	0.3743472337270094
(32000, 1)
CLICKBAIT (98%): How This Teacher Changed One Student's Life
(1, 5000)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7 stored elements and shape (1, 5000)>
  Coords	Values
  (0, 870)	0.48118225854341345
  (0, 2199)	0.2759974632596103
  (0, 2620)	0.32407181183677397
  (0, 3115)	0.3203684428662882
  (0, 4264)	0.4335248067107859
  (0, 4397)	0.49121901924685557
  (0, 4476)	0.23535223181884918
(32000, 1)
CLICKBAIT (100%): What Doctors Don't Tell You About Sleep
(1, 5000)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7 stored elements and shape (1, 5000)>
  Coords	Values
  (0, 130)	0